# Exploratory Analysis: Sampling Pairs Across Datasets

This notebook analyzes each CDCR dataset to understand:
1. Number of mentions per dataset
2. Number of training pairs generated by the sampling algorithm
3. Dataset statistics (topics, mentions per topic, etc.)

**Goal**: Determine how many pairs the new algorithm creates for each dataset without topic filtering.

## Setup

In [ ]:
import sys
import os
import pandas as pd
import numpy as np
from collections import defaultdict
import time

# Add CL_COREF to path
sys.path.insert(0, os.path.join(os.getcwd(), 'CL_COREF'))

from dataset import CDCR_Dataset, CL_Dataset
from model import create_tokenizer
from sentence_transformers import SentenceTransformer

print("Imports successful!")

## Configuration

In [ ]:
# Dataset configuration
BASE_DATA_PATH = "CL_COREF/data"
CONTEXT_METHOD = "hsu"
SPECIAL_MENTION_TOKENS = ("[E]", "[/E]")

# Model configuration
LANGUAGE_MODEL = "roberta-large"
SENTENCE_TRANSFORMER_MODEL = "all-mpnet-base-v2"

# Datasets to analyze
DATASETS = [
    "CD2CR",
    "CEREC",
    "ECBplus",
    "ECBplusMETAm",
    "FCC",
    "GVC",
    "HyperCoref",
    "NP4E",
    "NewsWCL50",
    "NiDENT",
    "WECEng"
]

# Sampling configuration
ADDITIONAL_SAMPLING = False

print(f"Will analyze {len(DATASETS)} datasets")
print(f"Datasets: {', '.join(DATASETS)}")

## Helper Functions

In [ ]:
def analyze_cdcr_dataset(cdcr_dataset):
    """
    Analyze a CDCR_Dataset instance and return statistics.
    """
    stats = {
        'dataset_name': cdcr_dataset.name,
        'total_mentions': len(cdcr_dataset.mentions),
        'total_topics': len(cdcr_dataset.mentions_by_topic),
        'entity_mentions': len(cdcr_dataset.mentions_by_type.get('entity', [])),
        'event_mentions': len(cdcr_dataset.mentions_by_type.get('event', [])),
        'total_documents': len(cdcr_dataset.docs),
        'topics': sorted(list(cdcr_dataset.mentions_by_topic.keys())),
    }
    
    # Mentions per topic
    mentions_per_topic = [len(mentions) for mentions in cdcr_dataset.mentions_by_topic.values()]
    stats['avg_mentions_per_topic'] = np.mean(mentions_per_topic) if mentions_per_topic else 0
    stats['min_mentions_per_topic'] = min(mentions_per_topic) if mentions_per_topic else 0
    stats['max_mentions_per_topic'] = max(mentions_per_topic) if mentions_per_topic else 0
    
    # Cluster statistics from gold_clusters_df
    if not cdcr_dataset.gold_clusters_df.empty:
        stats['total_clusters'] = cdcr_dataset.gold_clusters_df['gold_cluster'].nunique()
        cluster_sizes = cdcr_dataset.gold_clusters_df.groupby('gold_cluster').size()
        stats['avg_cluster_size'] = cluster_sizes.mean()
        stats['singletons'] = (cluster_sizes == 1).sum()
    else:
        stats['total_clusters'] = 0
        stats['avg_cluster_size'] = 0
        stats['singletons'] = 0
    
    return stats

def print_dataset_stats(stats):
    """
    Pretty print dataset statistics.
    """
    print(f"\n{'='*60}")
    print(f"Dataset: {stats['dataset_name']}")
    print(f"{'='*60}")
    print(f"Total Mentions: {stats['total_mentions']:,}")
    print(f"  - Entity mentions: {stats['entity_mentions']:,}")
    print(f"  - Event mentions: {stats['event_mentions']:,}")
    print(f"\nTopics: {stats['total_topics']}")
    print(f"  - Topic IDs: {', '.join(map(str, stats['topics'][:10]))}{'...' if len(stats['topics']) > 10 else ''}")
    print(f"  - Avg mentions/topic: {stats['avg_mentions_per_topic']:.1f}")
    print(f"  - Min mentions/topic: {stats['min_mentions_per_topic']}")
    print(f"  - Max mentions/topic: {stats['max_mentions_per_topic']}")
    print(f"\nClusters: {stats['total_clusters']:,}")
    print(f"  - Avg cluster size: {stats['avg_cluster_size']:.2f}")
    print(f"  - Singletons: {stats['singletons']:,}")
    print(f"\nDocuments: {stats['total_documents']:,}")

print("Helper functions defined!")

## Initialize Models

We need the tokenizer and sentence transformer model for creating CL_Dataset instances.

In [ ]:
print("Loading tokenizer...")
special_tokens = [f"[{t}]" for t in ["E"]] + [f"[/{t}]" for t in ["E"]]
tokenizer = create_tokenizer(LANGUAGE_MODEL, special_tokens)
print(f"✓ Tokenizer loaded: {LANGUAGE_MODEL}")

print("\nLoading sentence transformer model...")
st_model = SentenceTransformer(SENTENCE_TRANSFORMER_MODEL)
print(f"✓ Sentence transformer loaded: {SENTENCE_TRANSFORMER_MODEL}")

## Analyze Each Dataset

For each dataset, we'll:
1. Load the train split as a CDCR_Dataset
2. Analyze basic statistics
3. Create a CL_Dataset to see how many training pairs are generated
4. Store results for comparison

In [ ]:
results = []
detailed_stats = {}

for dataset_name in DATASETS:
    print(f"\n{'#'*60}")
    print(f"# Processing: {dataset_name}")
    print(f"{'#'*60}")
    
    try:
        # Path to train split
        train_path = os.path.join(BASE_DATA_PATH, dataset_name, "train")
        
        if not os.path.exists(train_path):
            print(f"⚠ Warning: Train path does not exist: {train_path}")
            continue
        
        # Load as CDCR_Dataset first (for basic stats)
        print(f"\n[1/3] Loading CDCR_Dataset for {dataset_name}...")
        start_time = time.time()
        cdcr_dataset = CDCR_Dataset(
            name=dataset_name,
            data_path=train_path,
            context_method=CONTEXT_METHOD,
            special_mention_tokens=SPECIAL_MENTION_TOKENS,
            predicted_topics=False,
            topic_filter=None,  # No filtering
            max_mentions=None   # No limit
        )
        load_time = time.time() - start_time
        print(f"  ✓ Loaded in {load_time:.2f}s")
        
        # Skip if no mentions loaded
        if len(cdcr_dataset.mentions) == 0:
            print(f"⚠ Skipping {dataset_name}: No mentions found in train split")
            continue
        
        # Analyze CDCR dataset
        print(f"\n[2/3] Analyzing dataset statistics...")
        stats = analyze_cdcr_dataset(cdcr_dataset)
        stats['load_time'] = load_time
        print_dataset_stats(stats)
        
        # Create CL_Dataset to count pairs
        print(f"\n[3/3] Creating CL_Dataset and sampling pairs...")
        start_time = time.time()
        
        cl_dataset = CL_Dataset(
            name=f"{dataset_name}_train",
            dataset_configs=[{
                "name": dataset_name,
                "path": train_path,
                "topic_filter": None,
                "max_mentions": None,
                "predicted_topics": False
            }],
            context_method=CONTEXT_METHOD,
            tokenizer=tokenizer,
            special_mention_tokens=SPECIAL_MENTION_TOKENS,
            mode="train",
            additional_sampling=ADDITIONAL_SAMPLING,
            sentence_transformer_model=SENTENCE_TRANSFORMER_MODEL
        )
        
        sampling_time = time.time() - start_time
        
        # Count pairs
        num_pairs = len(cl_dataset.all_pairs)
        num_positive_pairs = sum(cl_dataset.all_labels)
        num_negative_pairs = num_pairs - num_positive_pairs
        
        stats['num_pairs'] = num_pairs
        stats['num_positive_pairs'] = num_positive_pairs
        stats['num_negative_pairs'] = num_negative_pairs
        stats['sampling_time'] = sampling_time
        stats['pairs_per_mention'] = num_pairs / stats['total_mentions'] if stats['total_mentions'] > 0 else 0
        
        print(f"  ✓ Sampled in {sampling_time:.2f}s")
        print(f"\n  Training Pairs: {num_pairs:,}")
        print(f"    - Positive pairs: {num_positive_pairs:,} ({num_positive_pairs/num_pairs*100:.1f}%)")
        print(f"    - Negative pairs: {num_negative_pairs:,} ({num_negative_pairs/num_pairs*100:.1f}%)")
        print(f"    - Pairs per mention: {stats['pairs_per_mention']:.2f}")
        
        # Store results
        results.append(stats)
        detailed_stats[dataset_name] = {
            'cdcr_dataset': cdcr_dataset,
            'stats': stats
        }
        
        print(f"\n✓ {dataset_name} analysis complete!")
        
    except Exception as e:
        print(f"\n✗ Error processing {dataset_name}: {str(e)}")
        import traceback
        traceback.print_exc()
        continue

print(f"\n\n{'='*60}")
print(f"Analysis complete! Processed {len(results)}/{len(DATASETS)} datasets.")
print(f"{'='*60}")

## Summary Results Table

In [ ]:
# Create summary DataFrame
if len(results) == 0:
    print("⚠ No datasets were successfully processed. Check the errors above.")
else:
    summary_df = pd.DataFrame(results)

    # Select key columns for display
    display_columns = [
        'dataset_name',
        'total_mentions',
        'total_topics',
        'total_clusters',
        'num_pairs',
        'num_positive_pairs',
        'num_negative_pairs',
        'pairs_per_mention',
        'load_time',
        'sampling_time'
    ]

    summary_display = summary_df[display_columns].copy()

    # Format for better display
    summary_display['load_time'] = summary_display['load_time'].apply(lambda x: f"{x:.2f}s")
    summary_display['sampling_time'] = summary_display['sampling_time'].apply(lambda x: f"{x:.2f}s")
    summary_display['pairs_per_mention'] = summary_display['pairs_per_mention'].apply(lambda x: f"{x:.2f}")

    print("\n" + "="*100)
    print("SUMMARY: Dataset Statistics and Pair Counts")
    print("="*100)
    print(summary_display.to_string(index=False))

    # Save to CSV
    output_path = "dataset_analysis_results.csv"
    summary_df.to_csv(output_path, index=False)
    print(f"\n✓ Results saved to: {output_path}")

## Visualizations

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (15, 10)

# Create subplots
fig, axes = plt.subplots(2, 2, figsize=(16, 12))

# Plot 1: Total mentions per dataset
ax1 = axes[0, 0]
summary_df.plot(x='dataset_name', y='total_mentions', kind='bar', ax=ax1, color='steelblue')
ax1.set_title('Total Mentions per Dataset', fontsize=14, fontweight='bold')
ax1.set_xlabel('Dataset', fontsize=12)
ax1.set_ylabel('Number of Mentions', fontsize=12)
ax1.tick_params(axis='x', rotation=45)
ax1.grid(axis='y', alpha=0.3)

# Plot 2: Training pairs per dataset
ax2 = axes[0, 1]
summary_df.plot(x='dataset_name', y='num_pairs', kind='bar', ax=ax2, color='coral')
ax2.set_title('Training Pairs per Dataset', fontsize=14, fontweight='bold')
ax2.set_xlabel('Dataset', fontsize=12)
ax2.set_ylabel('Number of Pairs', fontsize=12)
ax2.tick_params(axis='x', rotation=45)
ax2.grid(axis='y', alpha=0.3)

# Plot 3: Positive vs Negative pairs
ax3 = axes[1, 0]
x = np.arange(len(summary_df))
width = 0.35
ax3.bar(x - width/2, summary_df['num_positive_pairs'], width, label='Positive', color='green', alpha=0.7)
ax3.bar(x + width/2, summary_df['num_negative_pairs'], width, label='Negative', color='red', alpha=0.7)
ax3.set_title('Positive vs Negative Pairs per Dataset', fontsize=14, fontweight='bold')
ax3.set_xlabel('Dataset', fontsize=12)
ax3.set_ylabel('Number of Pairs', fontsize=12)
ax3.set_xticks(x)
ax3.set_xticklabels(summary_df['dataset_name'], rotation=45, ha='right')
ax3.legend()
ax3.grid(axis='y', alpha=0.3)

# Plot 4: Pairs per mention ratio
ax4 = axes[1, 1]
summary_df.plot(x='dataset_name', y='pairs_per_mention', kind='bar', ax=ax4, color='purple')
ax4.set_title('Pairs per Mention Ratio', fontsize=14, fontweight='bold')
ax4.set_xlabel('Dataset', fontsize=12)
ax4.set_ylabel('Pairs per Mention', fontsize=12)
ax4.tick_params(axis='x', rotation=45)
ax4.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig('dataset_analysis_plots.png', dpi=300, bbox_inches='tight')
print("✓ Plots saved to: dataset_analysis_plots.png")
plt.show()

## Detailed Analysis: Mentions and Clusters per Topic

In [ ]:
# Analyze mentions per topic for each dataset
for dataset_name, data in detailed_stats.items():
    cdcr_dataset = data['cdcr_dataset']
    
    print(f"\n{'='*60}")
    print(f"Topic Distribution: {dataset_name}")
    print(f"{'='*60}")
    
    # Create topic statistics
    topic_stats = []
    for topic_id, mention_ids in cdcr_dataset.mentions_by_topic.items():
        topic_stats.append({
            'topic_id': topic_id,
            'num_mentions': len(mention_ids)
        })
    
    topic_df = pd.DataFrame(topic_stats).sort_values('num_mentions', ascending=False)
    
    print(f"\nTop 10 topics by mention count:")
    print(topic_df.head(10).to_string(index=False))
    
    print(f"\nBottom 10 topics by mention count:")
    print(topic_df.tail(10).to_string(index=False))

## Key Findings and Recommendations

In [ ]:
print("\n" + "="*80)
print("KEY FINDINGS")
print("="*80)

# Total statistics
total_mentions = summary_df['total_mentions'].sum()
total_pairs = summary_df['num_pairs'].sum()
avg_pairs_per_mention = summary_df['pairs_per_mention'].mean()

print(f"\n1. OVERALL STATISTICS:")
print(f"   - Total datasets analyzed: {len(summary_df)}")
print(f"   - Total mentions across all datasets: {total_mentions:,}")
print(f"   - Total training pairs generated: {total_pairs:,}")
print(f"   - Average pairs per mention: {avg_pairs_per_mention:.2f}")

# Largest datasets
largest_by_mentions = summary_df.nlargest(3, 'total_mentions')
print(f"\n2. LARGEST DATASETS (by mentions):")
for idx, row in largest_by_mentions.iterrows():
    print(f"   - {row['dataset_name']}: {row['total_mentions']:,} mentions, {row['num_pairs']:,} pairs")

# Most pairs generated
most_pairs = summary_df.nlargest(3, 'num_pairs')
print(f"\n3. MOST TRAINING PAIRS GENERATED:")
for idx, row in most_pairs.iterrows():
    print(f"   - {row['dataset_name']}: {row['num_pairs']:,} pairs ({row['pairs_per_mention']:.2f} per mention)")

# Efficiency (pairs per mention)
most_efficient = summary_df.nlargest(3, 'pairs_per_mention')
print(f"\n4. HIGHEST PAIRS-PER-MENTION RATIO:")
for idx, row in most_efficient.iterrows():
    print(f"   - {row['dataset_name']}: {row['pairs_per_mention']:.2f} pairs/mention")

print(f"\n5. RECOMMENDATIONS:")
print(f"   - For balanced multi-dataset training, consider using max_mentions to equalize")
print(f"   - Datasets with high pairs-per-mention ratio may dominate training")
print(f"   - Consider topic filtering for very large datasets to reduce training time")
print(f"   - Check class balance (positive vs negative) for each dataset")

print(f"\n" + "="*80)

## Export Detailed Results

In [ ]:
# Export full statistics
full_stats_path = "dataset_analysis_full_stats.csv"
summary_df.to_csv(full_stats_path, index=False)
print(f"✓ Full statistics saved to: {full_stats_path}")

# Create a configuration suggestion file
print("\n" + "="*80)
print("SUGGESTED CONFIGURATIONS FOR data_setups.yaml")
print("="*80)

# Suggest balanced configurations
median_mentions = summary_df['total_mentions'].median()

print(f"\n# Balanced configuration (equalizing to median: {int(median_mentions)} mentions)")
for idx, row in summary_df.iterrows():
    if row['total_mentions'] > median_mentions * 1.5:
        suggested_max = int(median_mentions)
        print(f"\n{idx + 10}:")
        print(f"  name: \"{row['dataset_name'].lower()}_balanced\"")
        print(f"  datasets:")
        print(f"    - name: \"{row['dataset_name']}\"")
        print(f"      splits: [\"train\", \"val\"]")
        print(f"      topic_filter: null")
        print(f"      max_mentions: {suggested_max}  # Original: {row['total_mentions']:,}")

print("\n✓ Analysis complete!")